**Name:** Thiều Quang  
**ID:** 24127114
**Thành viên:** Tôn Thất Việt Khoa — 24127190  
**Môn học:** Khai thác dữ liệu và ứng dụng (*Data Mining and Applications*)  
**Đề bài:** [HCMUS — Khai thác tập mục phổ biến](https://hcmus-cs-ta.github.io/lab-book/data-mining/projects/itemset/)

# Đồ án Eclat — Notebook thực nghiệm

**Project:** Group_9 - Eclat (Python + Julia).
**Colab:** Python 3 runtime. Upload this notebook to Colab, then provide Group_9_Colab.zip in Step 1.
Run Steps 1-7, Step 7B (Julia), and finally Step 8 (download results).
**VS Code:** Run scripts via PowerShell in D:\Group_9.
Runtime measurements on different machines are not directly comparable.


## Bước 1 — Tìm thư mục dự án hoặc tải ZIP

Chạy ô dưới đây một lần. Khi Colab hiện nút **Choose Files**, chọn **toàn bộ ZIP** `Group_9_Colab.zip` vừa tải, **không chọn riêng** `.ipynb`. Nếu đang trong thư mục project của VS Code, ô sẽ tự phát hiện mã nguồn, không yêu cầu upload.

In [1]:
from pathlib import Path
import os
import sys
import zipfile
import importlib

base = Path("/content")
project = base / "Group_9"
zip_path = base / "Group_9_Colab.zip"

# Nếu project chưa được giải nén thì lấy từ ZIP
if not (project / "src/algorithm/eclat.py").is_file():
    if not zip_path.exists():
        print("Chọn file Group_9_Colab.zip")
        colab_files = importlib.import_module("google.colab.files")
        colab_files.upload()

    with zipfile.ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            destination = (base / member.filename).resolve()
            if not destination.is_relative_to(base.resolve()):
                raise ValueError("ZIP chứa đường dẫn không an toàn")
        archive.extractall(base)

# Xác nhận cấu trúc project
assert (project / "src/algorithm/eclat.py").exists(), \
    "Không tìm thấy mã nguồn Python Eclat!"

assert (project / "julia/EclatJulia.jl").exists(), \
    "Không tìm thấy mã nguồn Julia!"

os.chdir(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

print("✓ Đã tìm thấy project:", project)
print("✓ Python Eclat: OK")
print("✓ Julia Eclat: OK")
print("✓ Thư mục làm việc:", os.getcwd())

Chọn file Group_9_Colab.zip


Saving Group_9_Colab.zip to Group_9_Colab.zip
✓ Đã tìm thấy project: /content/Group_9
✓ Python Eclat: OK
✓ Julia Eclat: OK
✓ Thư mục làm việc: /content/Group_9


## Bước 2 — Cài thư viện và chạy kiểm thử

`pytest` là kiểm tra nội bộ trên tập toy; **không thay thế** bước so với SPMF. Nếu test lỗi thì dừng để sửa, không sử dụng kết quả benchmark.

In [2]:
import subprocess

def run_module(name, *args):
    command = [sys.executable, '-m', name, *map(str, args)]
    print('>>>', ' '.join(command), flush=True)
    subprocess.run(command, cwd=project, check=True)

subprocess.run([sys.executable,'-m','pip','-q','install','-r','requirements.txt'],check=True)
run_module('pytest','-q')
run_module('src.cli','data/toy/baskets.txt','--minsup','0.5','--backend','bitset','--output','results/toy_result.txt')
print((project/'results/toy_result.txt').read_text(encoding='utf-8'))

>>> /usr/bin/python3 -m pytest -q
>>> /usr/bin/python3 -m src.cli data/toy/baskets.txt --minsup 0.5 --backend bitset --output results/toy_result.txt
1 #SUP: 4
2 #SUP: 5
3 #SUP: 4
1 2 #SUP: 3
1 3 #SUP: 3
2 3 #SUP: 3



## Bước 3 — Tải dữ liệu benchmark và dữ liệu ứng dụng

Chương trình dùng các URL được ghi trong `DATA_SOURCES.md`, lưu manifest kèm SHA-256. Nếu mạng nguồn lỗi, bạn có thể tải thủ công tại [SPMF Datasets](https://www.philippe-fournier-viger.com/spmf/index.php?link=datasets.php) và [UCI Optdigits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), rồi đặt đúng tên file vào thư mục `data/` (xem tài liệu nguồn). Không được tự thay bằng dữ liệu tạo ngẫu nhiên rồi gọi là benchmark thật.

In [3]:
run_module('scripts.download_data')
run_module('scripts.dataset_statistics')
print((project/'data/download_manifest.json').read_text(encoding='utf-8')[:1600])

>>> /usr/bin/python3 -m scripts.download_data
>>> /usr/bin/python3 -m scripts.dataset_statistics
{
  "foodmart": {
    "source_url": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/foodmartFIM.txt",
    "acquisition_method": "existing_with_matching_manifest",
    "description": "SPMF public FIM dataset, real retail",
    "local_path": "data/benchmark/foodmart.txt",
    "size_bytes": 82840,
    "sha256": "8762f2000459e94ee166bd813763567b2b60dfb24970e1cffec497b23a694081",
    "recorded_utc": "2026-10-27T19:32:29.518603+00:00"
  },
  "c20d10k": {
    "source_url": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/c20d10k.txt",
    "acquisition_method": "existing_with_matching_manifest",
    "description": "SPMF synthetic FIM dataset",
    "local_path": "data/benchmark/c20d10k.txt",
    "size_bytes": 800000,
    "sha256": "1ec8dca5aedaad57239481b726f67bf11cd8f51532aad0a2b6cffc47ddcf9394",
    "recorded_utc": "2026-10-27T19:32:29.524368+00:00"
  },
  "t25i10d10k": {


## Bước 4 — Chạy benchmark trên bốn bộ dữ liệu

Chạy Eclat `set` và `bitset`. Output: `results/benchmark.csv` và `results/rss_memory.csv`. `status=invalid` nghĩa là phép đo chưa hoàn tất, không được báo cáo như kết quả hợp lệ. **Nếu hết RAM**, giảm tải minh bạch, điều chỉnh minsup trong script và ghi thay đổi vào báo cáo; không tự ghi số liệu.

In [4]:
run_module('scripts.benchmark')
run_module('scripts.rss_memory')
run_module('scripts.scalability')
run_module('scripts.length_experiment')
print('Đã tạo benchmark.csv, rss_memory.csv, scalability.csv, length_experiment.csv')

>>> /usr/bin/python3 -m scripts.benchmark
>>> /usr/bin/python3 -m scripts.rss_memory
>>> /usr/bin/python3 -m scripts.scalability
>>> /usr/bin/python3 -m scripts.length_experiment
Đã tạo benchmark.csv, rss_memory.csv, scalability.csv, length_experiment.csv


## Bước 5 — Đối chiếu từng tập phổ biến và support với SPMF

SPMF chỉ chạy như **phần mềm tham chiếu Java**, không dùng thư viện hoặc mã nguồn của nó để cài Eclat. Chương trình thử tải release `.jar` từ trang chính thức. Nếu bị lỗi mạng, tải file `spmf.jar` từ [SPMF Download](https://www.philippe-fournier-viger.com/spmf/index.php?link=download.php) và upload thủ công bằng ô dự phòng dưới đây.

In [5]:
jar = project/'tools/spmf.jar'
if not jar.exists():
    try:
        run_module('scripts.setup_spmf')
    except subprocess.CalledProcessError:
        print('Không tự tải được SPMF. Chạy ô tải file spmf.jar thủ công ngay dưới đây.')
print('Đã có SPMF jar?', jar.is_file())

Đã có SPMF jar? True


**Tệp JAR dự phòng:** chỉ chạy ô sau nếu bước trước chưa tải được `tools/spmf.jar`. Chọn JAR phát hành chính thức từ https://www.philippe-fournier-viger.com/spmf/index.php?link=download.php. Bỏ qua ô này nếu tệp đã có.

In [6]:
import importlib
# Chỉ chạy nếu SPMF jar chưa có trong thư mục tools/.
if not jar.is_file():
    if not Path('/content').is_dir():
        raise RuntimeError('Trên VS Code: chép file spmf.jar vào tools/spmf.jar, không dùng upload Colab.')
    upload_tool = importlib.import_module('google.colab.files')
    uploaded_jar = upload_tool.upload()
    choices = [name for name in uploaded_jar if name.lower().endswith('.jar')]
    if not choices:
        raise ValueError('Chưa chọn .jar chính thức')
    import shutil
    jar.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(choices[0], jar)
    print('Đã lưu:', jar)
else:
    print('Đã có SPMF JAR. Bỏ qua bước upload dự phòng.')

Đã có SPMF JAR. Bỏ qua bước upload dự phòng.


In [7]:
import os
import subprocess
from pathlib import Path

JAVA_HOME = Path("/content/java25")
JAVA_BIN = JAVA_HOME / "bin" / "java"

if not JAVA_BIN.exists():
    JAVA_HOME.mkdir(parents=True, exist_ok=True)

    url = (
        "https://api.adoptium.net/v3/binary/latest/25/"
        "ga/linux/x64/jdk/hotspot/normal/eclipse"
    )

    archive = "/content/java25.tar.gz"

    print("Dang tai Java 25...")

    subprocess.run(
        ["curl", "-fL", "--retry", "3", "-o", archive, url],
        check=True
    )

    subprocess.run(
        [
            "tar", "-xzf", archive,
            "-C", str(JAVA_HOME),
            "--strip-components=1"
        ],
        check=True
    )

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = str(JAVA_HOME / "bin") + ":" + os.environ["PATH"]

print("=== JAVA VERSION ===")
subprocess.run(["java", "-version"], check=True)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("JAVA 25 READY")

Dang tai Java 25...
=== JAVA VERSION ===
JAVA_HOME: /content/java25
JAVA 25 READY


In [8]:
import csv
assert jar.is_file(), 'Missing tools/spmf.jar'
run_module('scripts.run_spmf_fixed', '--jar', 'tools/spmf.jar')
with (project / 'results/spmf_comparison.csv').open(
        encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
matched = [r for r in rows if r['status'] == 'ok'
           and r['exact_match'].lower() == 'true']
print(f'SPMF EXACT MATCH: {len(matched)}/{len(rows)}')
assert len(rows) == len(matched) == 25, 'SPMF comparison incomplete'


>>> /usr/bin/python3 -m scripts.run_spmf_fixed --jar tools/spmf.jar
SPMF EXACT MATCH: 25/25


## Bước 6 — Hai ứng dụng (Chương 3)

- Ứng dụng 1: ECommerce từ SPMF → giỏ hàng → luật support, confidence, lift → top 10.
- Ứng dụng 2: UCI Optdigits → patches → k-means NumPy → visual words → Eclat.

Đầu ra tạo bằng **mã Eclat của project**. Dữ liệu ứng dụng không được thay bằng toy khi báo cáo kết quả thật.

In [9]:
run_module('scripts.market_app')
run_module('scripts.visual_app')
for filename in ('market_top10.csv','visual_patterns.csv'):
    with (project/'results'/filename).open(encoding='utf-8-sig') as handle:
        print('\n',filename)
        print(''.join(handle.readlines()[:7]))

>>> /usr/bin/python3 -m scripts.market_app
>>> /usr/bin/python3 -m scripts.visual_app

 market_top10.csv
X,Y,support,confidence,lift
DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,0.01209,0.70155,33.88941
SPACEBOY LUNCH BOX,DOLLY GIRL LUNCH BOX,0.01209,0.58387,33.88941
PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,0.01449,0.65957,32.81438
GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.01449,0.72093,32.81438
GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,0.01449,0.52927,32.61643
PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.01449,0.893,32.61643


 visual_patterns.csv
visual_words,support_count,support_fraction
9,600,1.000000
6 9,448,0.746667
6,448,0.746667
0 9,404,0.673333
0,404,0.673333
9 11,389,0.648333



## Bước 7 — Tạo biểu đồ, bảng báo cáo và kiểm tra trạng thái

`docs/generated/experiment_data.tex` được sinh **từ CSV thực tế**; file này chưa tự thay tất cả phần nhận xét trong `docs/Report.tex`. Hãy giải thích kết quả của bạn, rồi biên dịch PDF tại VS Code khi có XeLaTeX.

In [10]:
run_module('scripts.plot_results')
run_module('scripts.generate_report_tables')
if not Path('/content').is_dir():
    run_module('scripts.check_submission')
else:
    print('Report.pdf is kept on Windows, not in the Colab ZIP')
print('Đường dẫn kết quả:',project/'results')

>>> /usr/bin/python3 -m scripts.plot_results
>>> /usr/bin/python3 -m scripts.generate_report_tables
Report.pdf is kept on Windows, not in the Colab ZIP
Đường dẫn kết quả: /content/Group_9/results


## Bước 7B — Kiểm chứng Eclat bằng Julia

Bên cạnh phiên bản Python, nhóm cài đặt độc lập thuật toán Eclat bằng Julia với hai cách biểu diễn tập TID: `Set` và `BitVector`.

Phần này kiểm thử chương trình Julia, chạy benchmark trên bốn bộ dữ liệu và đối chiếu chính xác từng frequent itemset cùng giá trị support với SPMF.

**Lưu ý:** Thời gian thực thi của Julia, Python và SPMF được đo theo những giao thức khác nhau. Do đó, kết quả chỉ mang tính tham khảo, không dùng để kết luận tốc độ thuần thuật toán khi chưa thống nhất điều kiện thực nghiệm.

In [11]:

from pathlib import Path
import os

workspace = Path("/content") if Path("/content").exists() else Path.cwd()

candidates = [
    workspace / "Group_9" / "julia" / "EclatJulia.jl",
    workspace / "julia" / "EclatJulia.jl",
    Path.cwd() / "julia" / "EclatJulia.jl",
]

found = next((p for p in candidates if p.exists()), None)

if found is None:
    found = next(
        (p for p in workspace.rglob("EclatJulia.jl")
         if ".venv" not in p.parts and "drive" not in p.parts),
        None
    )

if found is None:
    raise FileNotFoundError(
        "Chua tim thay julia/EclatJulia.jl. "
        "Hay chay buoc nap project truoc."
    )

ROOT = found.parent.parent.resolve()
os.chdir(ROOT)

print("Project:", ROOT)
print("Julia source:", found)


Project: /content/Group_9
Julia source: /content/Group_9/julia/EclatJulia.jl


In [12]:

from pathlib import Path
import shutil
import subprocess

julia_executable = shutil.which("julia")

if julia_executable is None:
    if not Path("/content").exists():
        raise RuntimeError(
            "Julia chua duoc cai. Tren Windows hay dung juliaup."
        )

    version = "1.13.3"
    url = (
        "https://julialang-s3.julialang.org/bin/linux/x64/1.13/"
        f"julia-{version}-linux-x86_64.tar.gz"
    )

    archive = Path("/content") / f"julia-{version}.tar.gz"
    installed = Path("/content") / f"julia-{version}" / "bin" / "julia"

    if not installed.exists():
        print("Installing Julia for Colab...")
        subprocess.run(
            ["wget", "-q", "-O", str(archive), url],
            check=True
        )
        subprocess.run(
            ["tar", "-xzf", str(archive), "-C", "/content"],
            check=True
        )

    julia_executable = str(installed)

if julia_executable is None:
    raise RuntimeError("Tidak dapat menemukan executable Julia.")

JULIA: str = julia_executable
subprocess.run([JULIA, "--version"], check=True)


CompletedProcess(args=['/usr/local/bin/julia', '--version'], returncode=0)

In [13]:

import subprocess

print("=== JULIA CORRECTNESS TESTS ===")

subprocess.run(
    [JULIA, "--startup-file=no", "julia/test_eclat.jl"],
    cwd=ROOT,
    check=True
)


=== JULIA CORRECTNESS TESTS ===


CompletedProcess(args=['/usr/local/bin/julia', '--startup-file=no', 'julia/test_eclat.jl'], returncode=0)

In [14]:

import subprocess
import sys

print("=== JULIA BENCHMARK ===")

subprocess.run(
    [JULIA, "--startup-file=no", "julia/benchmark_julia.jl"],
    cwd=ROOT,
    check=True
)

print("=== EXPORT JULIA ITEMSETS ===")

subprocess.run(
    [JULIA, "--startup-file=no", "julia/export_itemsets.jl"],
    cwd=ROOT,
    check=True
)

print("=== JULIA VS SPMF EXACT MATCH ===")

subprocess.run(
    [sys.executable, "-m", "scripts.verify_julia_spmf"],
    cwd=ROOT,
    check=True
)

import json, platform
from datetime import datetime, timezone
info = {
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'platform': platform.platform(),
    'julia': subprocess.check_output([JULIA, '--version'], text=True).strip(),
    'colab': Path('/content').is_dir()
}
(ROOT / 'results/julia_execution_environment.json').write_text(
    json.dumps(info, indent=2), encoding='utf-8')
print('Julia environment:', info)


=== JULIA BENCHMARK ===
=== EXPORT JULIA ITEMSETS ===
=== JULIA VS SPMF EXACT MATCH ===
Julia environment: {'run_utc': '2026-10-27T19:34:14.177962+00:00', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'julia': 'julia version 1.12.6', 'colab': True}


## Bước 8 — Tải kết quả thực nghiệm về máy

Chạy ô này để nhận `Eclat_Ket_Qua_Thuc_Nghiem_Colab.zip`. ZIP chứa thư mục `results/`, file manifest và bảng TeX. **Không chứa dữ liệu thô**. Giải nén vào đúng thư mục project đang mở bằng VS Code, rồi kiểm tra CSV và cập nhật nhận xét trong báo cáo. Không tự coi kết quả trên Colab là kết quả trên Windows nếu cấu hình phần cứng khác.

In [15]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import csv, unicodedata

with (project / 'results/spmf_comparison.csv').open(
        encoding='utf-8-sig', newline='') as f:
    spmf = list(csv.DictReader(f))
assert len(spmf) == 25 and all(
    r['status'] == 'ok' and r['exact_match'].lower() == 'true'
    for r in spmf), 'SPMF must be 25/25'

with (project / 'results/julia_benchmark.csv').open(
        encoding='utf-8-sig', newline='') as f:
    julia_rows = list(csv.DictReader(f))
assert len(julia_rows) == 48 and all(
    r['count_match'].lower() == 'true'
    and r['backend_exact_match'].lower() == 'true'
    for r in julia_rows), 'Julia must be 48/48'

readme = project / 'results/README.md'
if readme.is_file():
    content = readme.read_text(encoding='utf-8')
    updated = []
    for line in content.splitlines():
        plain = ''.join(x for x in unicodedata.normalize('NFD', line.lower())
                        if unicodedata.category(x) != 'Mn').replace('đ', 'd')
        if 'bms_webview_1' in plain and 'trang thai loi' in plain:
            updated.append('SPMF Colab: 25/25 exact matches, including BMS_WebView_1 with ID remapping.')
        else:
            updated.append(line)
    readme.write_text('\n'.join(updated) + '\n', encoding='utf-8')

output = base / 'Eclat_Ket_Qua_Thuc_Nghiem_Colab.zip'
collected = [x for x in (project / 'results').rglob('*') if x.is_file()]
for rel in ('data/download_manifest.json', 'docs/generated/experiment_data.tex'):
    file = project / rel
    if file.is_file():
        collected.append(file)
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for file in collected:
        archive.write(file, file.relative_to(project))
print('Created:', output)
if Path('/content').is_dir():
    colab_files = importlib.import_module("google.colab.files")
    colab_files.download(str(output))


Created: /content/Eclat_Ket_Qua_Thuc_Nghiem_Colab.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>